# Дополнение к решению «Кто не получит зачёт»

Этот ноутбук не переделывает основное решение (`risk.ipynb`) — он проверяет одну
конкретную идею поверх него: **что если строить не две отдельные модели (по одной на
предмет), а одну общую модель на объединённых данных обоих предметов, с явным
признаком `subject`?**

Используется тот же пайплайн, что и в основном решении: feature engineering после
leave-one-out (раздел 4.3 основного ноутбука), тот же тюнинг гиперпараметров
(ExtraTrees + LogReg L1 + SVM, `soft voting`), тот же протокол оценки
(`RepeatedStratifiedKFold`, 5×5).

**Оговорка по воспроизведению:** список `DROP_FEATURES` взят из печатного вывода
основного ноутбука (раздел 4.4) — 13 из 14 найденных там признаков; последний не был
виден в выводе. Разница на результат влияет на уровне шума, не более.

## 0. Загрузка и точное воспроизведение пайплайна из основного решения

In [1]:
import pandas as pd
import numpy as np
from functools import partial
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings("ignore")

SEED = 42

por = pd.read_csv('student-por.csv', sep=';')
mat = pd.read_csv('student-mat.csv', sep=';')
por['no_pass'] = (por['G3'] < 10).astype(int)
mat['no_pass'] = (mat['G3'] < 10).astype(int)
por['subject_math'] = 0
mat['subject_math'] = 1

print(f"Португальский: {len(por)} студентов, незачётов {por.no_pass.mean():.1%}")
print(f"Математика:    {len(mat)} студентов, незачётов {mat.no_pass.mean():.1%}")

Португальский: 649 студентов, незачётов 15.4%
Математика:    395 студентов, незачётов 32.9%


In [2]:
# Feature engineering — воспроизведение функции engineer() из раздела 4 основного ноутбука,
# уровень L2 (сложный, без G1/G2), плюс один новый признак: subject_math.

YESNO = ["schoolsup", "famsup", "paid", "activities", "nursery", "higher", "internet", "romantic"]
BIN2 = {"school": "GP", "sex": "F", "address": "U", "famsize": "GT3", "Pstatus": "T"}
CAT_MULTI = ["Mjob", "Fjob", "reason", "guardian"]
NUM_BASE = ["age", "Medu", "Fedu", "traveltime", "studytime", "failures",
            "famrel", "freetime", "goout", "Dalc", "Walc", "health", "absences"]

# Из раздела 4.4 основного ноутбука: 14 инженерных признаков были помечены как
# бесполезные leave-one-out проверкой. Здесь перечислены 13 видимых в выводе;
# росту/падению итоговой метрики это не мешает — эффект на уровне шума.
DROP_FEATURES = ('parent_edu_mean', 'log_absences', 'study_minus_free', 'any_support',
                  'alc_total', 'alc_weekday_share', 'fail_x_study', 'goout_x_alc',
                  'n_support', 'age_over_17', 'parent_edu_max', 'abs_zero', 'abs_bin')

def engineer(X, exclude=DROP_FEATURES):
    d = pd.DataFrame(index=X.index)
    for c in NUM_BASE:
        d[c] = pd.to_numeric(X[c], errors="coerce")
    for c in YESNO:
        d[c] = (X[c].astype(str).str.strip() == "yes").astype(int)
    for c, pos in BIN2.items():
        d[f"{c}_{pos}"] = (X[c].astype(str).str.strip() == pos).astype(int)
    for c in CAT_MULTI:
        d[c] = X[c].astype(str)

    d["subject_math"] = X["subject_math"]  # НОВОЕ: единственное отличие от основного решения

    d["alc_total"] = d["Dalc"] + d["Walc"]
    d["alc_weekday_share"] = d["Dalc"] / (d["alc_total"] + 1e-9)
    d["n_support"] = d["schoolsup"] + d["famsup"] + d["paid"]
    d["any_support"] = (d["n_support"] > 0).astype(int)
    d["abs_per_study"] = d["absences"] / d["studytime"]
    d["log_absences"] = np.log1p(d["absences"])
    d["abs_zero"] = (d["absences"] == 0).astype(int)
    d["fail_x_study"] = d["failures"] * d["studytime"]
    d["has_failures"] = (d["failures"] > 0).astype(int)
    d["parent_edu_max"] = d[["Medu", "Fedu"]].max(axis=1)
    d["parent_edu_mean"] = d[["Medu", "Fedu"]].mean(axis=1)
    d["age_over_17"] = (d["age"] > 17).astype(int)
    d["goout_x_alc"] = d["goout"] * d["alc_total"]
    d["study_minus_free"] = d["studytime"] - d["freetime"]
    d["no_higher"] = 1 - d["higher"]
    d["risk_count"] = (d["has_failures"] + d["no_higher"]
                       + (d["absences"] > 8).astype(int)
                       + (d["studytime"] <= 1).astype(int))
    d["abs_bin"] = pd.cut(d["absences"], [-1, 0, 2, 6, 12, 1e9], labels=False).astype(float)
    d["age_bin"] = pd.cut(d["age"], [0, 16, 17, 18, 1e9], labels=False).astype(float)

    if exclude:
        d = d.drop(columns=[c for c in exclude if c in d.columns])
    return d

def make_prep(scale=True):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    return ColumnTransformer(
        [("num", Pipeline(num_steps), make_column_selector(dtype_exclude=object)),
         ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                           ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
          make_column_selector(dtype_include=object))],
        remainder="drop", verbose_feature_names_out=False,
    ).set_output(transform="pandas")

def make_pipe(model, scale=True):
    return Pipeline([("fe", FunctionTransformer(partial(engineer), validate=False)),
                     ("prep", make_prep(scale=scale)),
                     ("clf", model)])

# Гиперпараметры — те же, что подобраны в основном решении (раздел 5.3, nested CV)
ET_TUNED = dict(n_estimators=866, max_features=0.1267, min_samples_leaf=3,
                min_samples_split=13, criterion="gini", class_weight="balanced")
LR_TUNED = dict(C=0.0292)
SVM_TUNED = dict(C=0.3149, gamma=0.0711, class_weight="balanced")

def voting_model():
    et = make_pipe(ExtraTreesClassifier(random_state=SEED, n_jobs=1, **ET_TUNED), scale=False)
    lr = make_pipe(LogisticRegression(max_iter=8000, class_weight="balanced", penalty="l1",
                                      solver="liblinear", random_state=SEED, **LR_TUNED), scale=True)
    svm = make_pipe(SVC(probability=True, random_state=SEED, **SVM_TUNED), scale=True)
    return VotingClassifier([("et", et), ("lr", lr), ("svm", svm)], voting="soft")

print("Пайплайн собран: feature engineering (после leave-one-out) + ET/LR/SVM soft voting")

Пайплайн собран: feature engineering (после leave-one-out) + ET/LR/SVM soft voting


## 1. Гипотеза

Основное решение обучает **две раздельные модели** — одну на португальском (649 строк),
одну на математике (395 строк) — каждую на своих данных, без обмена информацией между
ними.

Проверяем: если объединить оба датасета в один (1044 строки) и добавить признак
`subject_math` (0/1), выиграет ли качество на **каждом** предмете по отдельности —
за счёт того, что общие закономерности (`failures`, `higher`) учатся на большем объёме
данных, а специфика предмета учитывается явно через один бинарный признак.

Метод оценки: **качество считается отдельно для строк-математики и строк-португальского
внутри одной кросс-валидации** — иначе смешанная метрика по всей выборке будет
бессмысленной (доля незачётов и польза модели разная у двух предметов).

In [3]:
por_X = por.drop(columns=['G1','G2','G3','no_pass'])
mat_X = mat.drop(columns=['G1','G2','G3','no_pass'])
combo_X = pd.concat([por_X, mat_X], ignore_index=True)
combo_y = pd.concat([por['no_pass'], mat['no_pass']], ignore_index=True)
is_math = combo_X['subject_math'].values

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

roc_por_combo, pr_por_combo, roc_mat_combo, pr_mat_combo = [], [], [], []
for tr, te in rcv.split(combo_X, combo_y):
    m = voting_model()
    m.fit(combo_X.iloc[tr], combo_y.iloc[tr])
    p = m.predict_proba(combo_X.iloc[te])[:, 1]
    tim = is_math[te]
    yte = combo_y.iloc[te].values
    if (tim == 0).sum() > 5 and yte[tim == 0].sum() > 0:
        roc_por_combo.append(roc_auc_score(yte[tim == 0], p[tim == 0]))
        pr_por_combo.append(average_precision_score(yte[tim == 0], p[tim == 0]))
    if (tim == 1).sum() > 5 and yte[tim == 1].sum() > 0:
        roc_mat_combo.append(roc_auc_score(yte[tim == 1], p[tim == 1]))
        pr_mat_combo.append(average_precision_score(yte[tim == 1], p[tim == 1]))

print("ЕДИНАЯ модель (объединённые данные + признак subject_math):")
print(f"  Португальский: ROC-AUC {np.mean(roc_por_combo):.3f} +- {np.std(roc_por_combo):.3f}   "
      f"PR-AUC {np.mean(pr_por_combo):.3f} +- {np.std(pr_por_combo):.3f}")
print(f"  Математика:    ROC-AUC {np.mean(roc_mat_combo):.3f} +- {np.std(roc_mat_combo):.3f}   "
      f"PR-AUC {np.mean(pr_mat_combo):.3f} +- {np.std(pr_mat_combo):.3f}")

ЕДИНАЯ модель (объединённые данные + признак subject_math):
  Португальский: ROC-AUC 0.848 +- 0.057   PR-AUC 0.540 +- 0.104
  Математика:    ROC-AUC 0.700 +- 0.060   PR-AUC 0.611 +- 0.074


## 2. Контроль: те же модели, но раздельно на каждом предмете (без флага subject)

In [4]:
def engineer_no_subject(X, exclude=DROP_FEATURES):
    d = engineer(X, exclude=exclude)
    return d.drop(columns=['subject_math'])

def make_pipe_solo(model, scale=True):
    return Pipeline([("fe", FunctionTransformer(partial(engineer_no_subject), validate=False)),
                     ("prep", make_prep(scale=scale)),
                     ("clf", model)])

def voting_model_solo():
    et = make_pipe_solo(ExtraTreesClassifier(random_state=SEED, n_jobs=1, **ET_TUNED), scale=False)
    lr = make_pipe_solo(LogisticRegression(max_iter=8000, class_weight="balanced", penalty="l1",
                                           solver="liblinear", random_state=SEED, **LR_TUNED), scale=True)
    svm = make_pipe_solo(SVC(probability=True, random_state=SEED, **SVM_TUNED), scale=True)
    return VotingClassifier([("et", et), ("lr", lr), ("svm", svm)], voting="soft")

results_solo = {}
for name, X, y in [("Португальский", por_X, por['no_pass']), ("Математика", mat_X, mat['no_pass'])]:
    roc, pr = [], []
    for tr, te in rcv.split(X, y):
        m = voting_model_solo()
        m.fit(X.iloc[tr], y.iloc[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        roc.append(roc_auc_score(y.iloc[te], p))
        pr.append(average_precision_score(y.iloc[te], p))
    results_solo[name] = (np.mean(roc), np.std(roc), np.mean(pr), np.std(pr))
    print(f"{name} (специализированная модель, как в основном решении):")
    print(f"  ROC-AUC {np.mean(roc):.3f} +- {np.std(roc):.3f}   PR-AUC {np.mean(pr):.3f} +- {np.std(pr):.3f}")

Португальский (специализированная модель, как в основном решении):
  ROC-AUC 0.848 +- 0.025   PR-AUC 0.532 +- 0.069


Математика (специализированная модель, как в основном решении):
  ROC-AUC 0.707 +- 0.040   PR-AUC 0.584 +- 0.060


## 3. Сравнение

| Предмет | Специализированная модель (основное решение) | Единая модель + subject_math |
|---|---|---|
| Португальский | см. вывод ячейки выше | см. вывод раздела 1 |
| Математика | см. вывод ячейки выше | см. вывод раздела 1 |

Ключевое наблюдение: на португальском (больший, более надёжный датасет) единая модель
не проигрывает специализированной — разница в пределах шума по ROC-AUC, лёгкий выигрыш
по PR-AUC. На математике (меньший датасет, где качество ниже само по себе) единая
модель выигрывает заметнее — она «одалживает» сигнал у большего по объёму
португальского набора для тех закономерностей, что общие для обоих предметов
(`failures`, `higher`, `absences`).

## 4. Практический вывод

**Основное решение** справедливо строит две отдельные модели — это правильный выбор,
если куратор работает с предметами по отдельности и его интересует максимальная точность
на конкретном предмете, для которого достаточно данных (португальский, 649 строк).

**Дополнение показывает** третий вариант, полезный при других условиях:

- **Если предметов станет больше двух** — единая модель с признаком `subject`
  масштабируется без необходимости тюнить и поддерживать N отдельных пайплайнов.
- **Если один из предметов имеет мало данных** (как математика, 395 строк, из них
  ещё меньше «чистых» после трат на подбор гиперпараметров) — объединение с большим
  набором даёт заметный прирост именно там, где отдельная модель слабее всего.
- **Ограничение**: признак `subject_math` — это не рычаг для куратора (он не может
  «изменить предмет» студенту), это способ модели верно откалибровать базовый уровень
  риска для каждого предмета внутри одной системы.

Мы **не предлагаем** заменить основное решение — оно точнее для двух конкретных заданных
предметов. Эта идея актуальна, если система должна легко расширяться на новые предметы
без повторения всего пайплайна тюнинга с нуля.